In [16]:
import ast
import json
from pathlib import Path

class SSATExtractorAST:
    """用Python内置的ast模块提取SSAT（避免pyreverse依赖）"""
    def __init__(self, file_path):
        self.file_path = Path(file_path)
        self.ssat = {
            "file_name": self.file_path.name,
            "classes": {}  # 存储类信息：类名→{方法, 参数}
        }

    def _parse_methods(self, class_node):
        """解析类中的方法及参数"""
        methods = []
        for node in class_node.body:
            if isinstance(node, ast.FunctionDef):  # 只处理函数/方法定义
                # 提取参数名（忽略self以外的特殊参数）
                params = []
                for param in node.args.args:
                    params.append(param.arg)  # 参数名（如"self", "a", "b"）
                methods.append({
                    "name": node.name,  # 方法名
                    "parameters": params  # 参数列表
                })
        return methods

    def extract(self):
        """主方法：解析代码生成SSAT"""
        # 读取代码内容
        with open(self.file_path, "r", encoding="utf-8") as f:
            code = f.read()

        # 解析抽象语法树
        tree = ast.parse(code)

        # 遍历树中的类定义
        for node in ast.walk(tree):
            if isinstance(node, ast.ClassDef):  # 找到类定义
                class_name = node.name  # 类名
                # 解析方法
                methods = self._parse_methods(node)
                # 依赖关系（单个文件暂为空）
                dependencies = []
                # 写入SSAT
                self.ssat["classes"][class_name] = {
                    "methods": methods,
                    "dependencies": dependencies
                }

        return self.ssat


# 测试：提取Python文件的SSAT
if __name__ == "__main__":
    # 动态查找input目录下的Python文件
    input_dir = Path("../input")
    python_files = list(input_dir.glob("*.py"))
    
    if not python_files:
        print("错误：在input目录中没有找到Python文件")
    elif len(python_files) > 1:
        print("错误：input目录中存在多个Python文件，请保证只有一个待重构文件")
    else:
        # 使用动态找到的文件
        extractor = SSATExtractorAST(file_path=python_files[0])
        ssat = extractor.extract()
        print("用ast提取的SSAT结构：")
        print(json.dumps(ssat, indent=2, ensure_ascii=False))

用ast提取的SSAT结构：
{
  "file_name": "messy_calculator.py",
  "classes": {
    "Calculator": {
      "methods": [
        {
          "name": "__init__",
          "parameters": [
            "self"
          ]
        },
        {
          "name": "add",
          "parameters": [
            "self",
            "a",
            "b"
          ]
        },
        {
          "name": "subtract",
          "parameters": [
            "self",
            "a",
            "b"
          ]
        },
        {
          "name": "multiply",
          "parameters": [
            "self",
            "a",
            "b"
          ]
        },
        {
          "name": "divide",
          "parameters": [
            "self",
            "a",
            "b"
          ]
        },
        {
          "name": "set_user",
          "parameters": [
            "self",
            "name",
            "email"
          ]
        },
        {
          "name": "get_user_info",
          "parameters": [
  

In [17]:
import os
import json
import ast
import sys
from pathlib import Path
from typing import List, Dict

# 添加utils目录到路径，以便导入api_client
sys.path.insert(0, str(Path(__file__).parent if '__file__' in globals() else Path.cwd()))

# 步骤1：SSAT提取器（正确无误）
class SSATExtractorAST:
    """用ast模块提取代码的SSAT结构"""
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.ssat = {
            "file_name": os.path.basename(file_path),
            "classes": {}
        }

    def extract(self) -> Dict:
        with open(self.file_path, "r", encoding="utf-8") as f:
            code = f.read()
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.ClassDef):
                class_name = node.name
                methods = []
                for func_node in node.body:
                    if isinstance(func_node, ast.FunctionDef):
                        params = [param.arg for param in func_node.args.args]
                        methods.append({"name": func_node.name, "parameters": params})
                self.ssat["classes"][class_name] = {"methods": methods}
        return self.ssat


# 步骤2：使用改进的DeepSeek API客户端（带重试和超时处理）
try:
    from utils.api_client import DeepSeekAPI
    print("已导入改进的API客户端（带重试机制和超时处理）")
except ImportError:
    # 如果导入失败，使用本地定义（向后兼容）
    import requests
    print("警告：无法导入改进的API客户端，使用基础版本")
    
    class DeepSeekAPI:
        """封装DeepSeek API调用（基础版本，带改进的超时和重试）"""
        def __init__(self, api_key: str = None, model: str = "deepseek-chat"):
            self.api_key = api_key or os.getenv("DEEPSEEK")
            if not self.api_key:
                raise ValueError("请设置环境变量DEEPSEEK或传入api_key参数")
            self.model = model
            self.api_url = "https://api.deepseek.com/v1/chat/completions"
            self.headers = {
                "Content-Type": "application/json",
                "Authorization": f"Bearer {self.api_key}"
            }

        def send_request(self, prompt: str, max_tokens: int = 2048) -> str:
            """发送API请求，带重试机制"""
            payload = {
                "model": self.model,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0.2,
                "max_tokens": max_tokens
            }
            
            max_retries = 3
            for attempt in range(max_retries):
                try:
                    # 使用更长的超时时间：(连接超时10秒, 读取超时120秒)
                    response = requests.post(
                        self.api_url,
                        headers=self.headers,
                        json=payload,
                        timeout=(10, 120)  # (连接超时, 读取超时)
                    )
                    response.raise_for_status()
                    return response.json()["choices"][0]["message"]["content"]
                except requests.exceptions.Timeout as e:
                    if attempt < max_retries - 1:
                        wait_time = 2 * (2 ** attempt)  # 指数退避：2秒, 4秒, 8秒
                        print(f"请求超时，{wait_time}秒后重试 (尝试 {attempt + 1}/{max_retries})...")
                        import time
                        time.sleep(wait_time)
                        continue
                    raise RuntimeError(f"API调用超时（已重试{max_retries}次）: {str(e)}")
                except requests.exceptions.RequestException as e:
                    if attempt < max_retries - 1:
                        wait_time = 2 * (2 ** attempt)
                        print(f"请求失败，{wait_time}秒后重试 (尝试 {attempt + 1}/{max_retries})...")
                        import time
                        time.sleep(wait_time)
                        continue
                    raise RuntimeError(f"API调用失败（已重试{max_retries}次）: {str(e)}")


# 步骤3：重构机会识别函数（修复位置和缩进）
def detect_refactor_opportunities(code_path: str) -> List[Dict]:  # 移到类外部，独立函数
    """完整流程：提取SSAT → 构建提示词 → 调用API"""
    # 1. 提取SSAT
    extractor = SSATExtractorAST(file_path=code_path)
    ssat = extractor.extract()

    # 2. 读取原始代码
    with open(code_path, "r", encoding="utf-8") as f:
        code = f.read()

    # 3. 构建完整的prompt（修复字符串闭合问题）
    ideal_constraints = {
        "单一职责原则": "一个类只负责一项核心功能（如计算类不应同时处理日志）",
        "无重复代码": "相同逻辑（如日志写入）应封装为一个方法，避免重复",
        "函数简洁性": "一个函数只做一件事，避免包含多个独立步骤"
    }

    # 关键修复：完整闭合字符串，包含所有必要部分
    prompt = f"""
    请分析以下代码的架构问题，基于SSAT和理想约束，返回重构机会：

    1. 原始代码：
    ```python
    {code}
    ```  # 原始代码部分结束

    2. 代码架构SSAT（类→方法→参数）：
    {json.dumps(ssat, indent=2)}

    3. 理想约束：
    {json.dumps(ideal_constraints, indent=2)}

    输出要求：JSON列表，每个元素包含"问题类型"、"位置"、"问题描述"、"建议方案"。
    """  # 这里添加结尾的三引号，闭合字符串


    # 4. 调用DeepSeek API
    api = DeepSeekAPI()
    response = api.send_request(prompt=prompt)

    # 5. 解析结果
    if "```json" in response:
        response = response.split("```json")[1].split("```")[0].strip()
    return json.loads(response)


# 运行测试
if __name__ == "__main__":
    try:
        # 动态查找input目录下的Python文件
        input_dir = Path("../input")
        python_files = list(input_dir.glob("*.py"))
        
        if not python_files:
            print("错误：在input目录中没有找到Python文件")
        elif len(python_files) > 1:
            print("错误：input目录中存在多个Python文件，请保证只有一个待重构文件")
        else:
            # 确保当前目录有messy_calculator.py文件
            opportunities = detect_refactor_opportunities(code_path=str(python_files[0]))
            print("识别到的重构机会：")
            print(json.dumps(opportunities, indent=2, ensure_ascii=False))
    except Exception as e:
        print(f"运行出错：{e}")

警告：无法导入改进的API客户端，使用基础版本
识别到的重构机会：
[
  {
    "问题类型": "违反单一职责原则",
    "位置": "Calculator类",
    "问题描述": "Calculator类承担了过多职责，包括计算功能、用户管理、数据统计、文件操作、格式化输出等，违反了单一职责原则",
    "建议方案": "将Calculator类拆分为多个单一职责的类：Calculator（核心计算）、UserManager（用户管理）、StatisticsCalculator（数据统计）、HistoryManager（历史记录管理）、FileExporter（文件导出）"
  },
  {
    "问题类型": "代码重复",
    "位置": "add、subtract、multiply、divide方法",
    "问题描述": "四个计算方法中存在大量重复代码：结果赋值、历史记录添加、日志写入等逻辑在每个方法中重复出现",
    "建议方案": "提取公共的日志记录和历史记录逻辑到单独的方法中，如_log_operation()，然后在每个计算方法中调用"
  },
  {
    "问题类型": "函数过长且职责过多",
    "位置": "process_calculation_request方法",
    "问题描述": "该方法承担了请求解析、计算执行、日志记录、结果检查等多个职责，函数过长且复杂",
    "建议方案": "拆分为多个方法：_parse_request()、_execute_calculation()、_check_result_anomaly()，每个方法只负责单一功能"
  },
  {
    "问题类型": "违反单一职责原则",
    "位置": "calculate_statistics方法",
    "问题描述": "该方法在Calculator类中计算统计信息，与计算器的核心职责无关",
    "建议方案": "将统计计算功能提取到独立的StatisticsCalculator类中"
  },
  {
    "问题类型": "违反单一职责原则",
    "位置": "set_user、get_user_info方法",
    "问题描述": "用户管理功能与计算器核

In [18]:
import os
import json
import ast
import requests
from typing import List, Dict
from pathlib import Path

class SSATExtractorAST:
    def __init__(self, file_path: str):
        self.file_path = Path(file_path)
        self.ssat = {"file_name": os.path.basename(file_path), "classes": {}}

    def extract(self) -> Dict:
        with open(self.file_path, "r", encoding="utf-8") as f:
            code = f.read()
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.ClassDef):
                class_name = node.name
                methods = []
                for func_node in node.body:
                    if isinstance(func_node, ast.FunctionDef):
                        params = [param.arg for param in func_node.args.args]
                        methods.append({"name": func_node.name, "parameters": params})
                self.ssat["classes"][class_name] = {"methods": methods}
        return self.ssat

class DeepSeekAPI:
    def __init__(self, api_key: str = None, model: str = "deepseek-chat"):
        self.api_key = api_key or os.getenv("DEEPSEEK")
        if not self.api_key:
            raise ValueError("请设置环境变量DEEPSEEKAPI")
        self.model = model
        self.api_url = "https://api.deepseek.com/v1/chat/completions"
        self.headers = {"Content-Type": "application/json", "Authorization": f"Bearer {self.api_key}"}

    def send_request(self, prompt: str) -> str:
        payload = {"model": self.model, "messages": [{"role": "user", "content": prompt}], "temperature": 0.2, "max_tokens": 2048}
        max_retries = 3
        for attempt in range(max_retries):
            try:
                # 使用更长的超时时间：(连接超时10秒, 读取超时120秒)
                response = requests.post(self.api_url, headers=self.headers, json=payload, timeout=(10, 120))
                response.raise_for_status()
                return response.json()["choices"][0]["message"]["content"]
            except requests.exceptions.Timeout as e:
                if attempt < max_retries - 1:
                    wait_time = 2 * (2 ** attempt)  # 指数退避：2秒, 4秒, 8秒
                    print(f"请求超时，{wait_time}秒后重试 (尝试 {attempt + 1}/{max_retries})...")
                    import time
                    time.sleep(wait_time)
                    continue
                raise RuntimeError(f"API调用超时（已重试{max_retries}次）: {str(e)}")
            except requests.exceptions.RequestException as e:
                if attempt < max_retries - 1:
                    wait_time = 2 * (2 ** attempt)
                    print(f"请求失败，{wait_time}秒后重试 (尝试 {attempt + 1}/{max_retries})...")
                    import time
                    time.sleep(wait_time)
                    continue
                raise RuntimeError(f"API调用失败（已重试{max_retries}次）: {str(e)}")

def detect_refactor_opportunities(code_path: str) -> List[Dict]:
    extractor = SSATExtractorAST(file_path=code_path)
    ssat = extractor.extract()
    with open(code_path, "r", encoding="utf-8") as f:
        code = f.read()
    ideal_constraints = {
        "单一职责原则": "一个类只负责一项核心功能",
        "无重复代码": "相同逻辑需封装为单一方法",
        "函数简洁性": "函数应只做一件事"
    }
    prompt = f"""分析以下代码的架构问题，基于SSAT和理想约束返回重构机会：
    1. 原始代码：```python{code}```
    2. SSAT：{json.dumps(ssat, indent=2)}
    3. 约束：{json.dumps(ideal_constraints, indent=2)}
    输出JSON列表，包含"问题类型"、"位置"、"问题描述"、"建议方案"。"""
    api = DeepSeekAPI()
    response = api.send_request(prompt=prompt)
    if "```json" in response:
        response = response.split("```json")[1].split("```")[0].strip()
    return json.loads(response)

class AutoRefactor:
    def __init__(self, api_key: str = None):
        self.api = DeepSeekAPI(api_key=api_key)

    def generate_refactored_code(self, original_code: str, opportunities: List[Dict]) -> str:
        prompt = f"""根据原始代码和重构机会，输出完整重构后代码（仅Python代码，放```python```块中）：
        1. 原始代码：```python{original_code}```
        2. 重构机会：{json.dumps(opportunities, indent=2)}
        要求：可运行，功能不变，修复架构问题，单文件结构。"""
        response = self.api.send_request(prompt=prompt)
        if "```python" in response:
            return response.split("```python")[1].split("```")[0].strip()
        return response

def full_auto_refactor(input_file: str, output_file: str):
    with open(input_file, "r", encoding="utf-8") as f:
        original_code = f.read()
    opportunities = detect_refactor_opportunities(input_file)
    refactor = AutoRefactor()
    refactored_code = refactor.generate_refactored_code(original_code, opportunities)
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(refactored_code)
    print(f"重构完成，保存至 {output_file}")
    return refactored_code

# 运行示例
if __name__ == "__main__":
    # 动态查找input目录下的Python文件
    input_dir = Path("../input")
    python_files = list(input_dir.glob("*.py"))
    
    if not python_files:
        print("错误：在input目录中没有找到Python文件")
    elif len(python_files) > 1:
        print("错误：input目录中存在多个Python文件，请保证只有一个待重构文件")
    else:
        full_auto_refactor(str(python_files[0]), "../output/refactored_calculator.py")

重构完成，保存至 ../output/refactored_calculator.py
